# JiT-RCDM Training on Google Colab — RETFound Encoder

Representation-Conditioned Diffusion Model — RETFound MAE ViT-Large/16 + JiT ViT + Flow Matching  
Dataset: Messidor-2 retinal fundus images (224×224)

This notebook mirrors `colab_training.ipynb` (DinoV3 ViT-S/16, 384-dim CLS
token), swapping the conditioning encoder for **RETFound** (ViT-Large/16 MAE,
1024-dim CLS token, pretrained on 1.6M retinal fundus images). Checkpoints are
saved to a separate `checkpoints_retfound/` folder on Drive so they don't
collide with the existing DinoV3 run.

## Before you start
1. **Runtime → Change runtime type → A100 GPU**
2. Upload the following to **Google Drive** under `MyDrive/jit_rcdm/`:
   - `train_packed_retfound.pt` — packed dataset (images + RETFound reps); create with `scripts/precompute_reps_retfound.py` + `scripts/pack_dataset.py` locally
   - `retfound/RETFound_mae_natureCFP.pth` — RETFound MAE ViT-L/16 checkpoint (~3.7 GB)
   - `test_images/` — a few `.png` fundus images for sampling (shared with the DinoV3 notebook)
3. Run cells **top to bottom**

## Workflow after local code changes
Edit locally → `git push origin main` → re-run **Cell 3** in Colab to pull latest → re-run **Cell 8** to reload imports.

## 1 — Check GPU

In [2]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {mem:.1f} GB")
    if mem >= 70:
        print("✓ A100/H100 80GB — use batch_size=256")
    elif mem >= 38:
        print("✓ A100 40GB — use batch_size=128")
    else:
        print("⚠ T4/V100 — use batch_size=32")
else:
    print("❌ No GPU — go to Runtime → Change runtime type → GPU")

PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
VRAM    : 42.4 GB
✓ A100 40GB — use batch_size=128


## 2 — Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/jit_rcdm"
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Drive mounted. Project folder: {DRIVE_DIR}")
print("Contents:")
for f in sorted(os.listdir(DRIVE_DIR)):
    full = os.path.join(DRIVE_DIR, f)
    tag  = f"({os.path.getsize(full)/1e6:.1f} MB)" if os.path.isfile(full) else "(folder)"
    print(f"  {f} {tag}")

Mounted at /content/drive
Drive mounted. Project folder: /content/drive/MyDrive/jit_rcdm
Contents:
  checkpoints (folder)
  checkpoints_retfound (folder)
  data (folder)
  dinov3_vits16_tmp (folder)
  retfound (folder)
  samples (folder)
  test_images (folder)
  train_packed.pt (147.8 MB)
  train_packed_retfound.pt (150.3 MB)
  train_reps.pt (1.6 MB)
  train_reps_retfound.pt (4.0 MB)


## 3 — Clone / pull JiT-RCDM code

> **After a `git push origin main`:** re-run this cell to pull the latest changes, then re-run Cell 8 to reload imports.

In [4]:
import os, sys

REPO_DIR = "/content/jit_rcdm"
REPO_URL = "https://github.com/SeverinLe/master_implementation.git"
BRANCH   = "main"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print(f"Cloning {BRANCH} branch ...")
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

%cd {REPO_DIR}

# Verify JiT-RCDM + RETFound encoder code is present
assert os.path.exists("rcdm/jit.py"),                  "❌ rcdm/jit.py missing — git pull may have failed"
assert os.path.exists("rcdm/encoder_retfound.py"),     "❌ rcdm/encoder_retfound.py missing — git pull may have failed"
assert os.path.exists("scripts/train.py"),             "❌ scripts/train.py missing"
assert os.path.exists("scripts/sampling_retfound.py"), "❌ scripts/sampling_retfound.py missing"
print("\n✓ JiT-RCDM + RETFound code present")
!ls rcdm/

Cloning main branch ...
Cloning into '/content/jit_rcdm'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 203 (delta 90), reused 103 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (203/203), 412.52 KiB | 14.73 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/content/jit_rcdm

✓ JiT-RCDM + RETFound code present
conditioning.py  encoder.py	      __init__.py
dataset.py	 encoder_retfound.py  jit.py


## 4 — Install dependencies

In [5]:
!pip install -q transformers safetensors wandb timm
print("✓ Dependencies installed")

from transformers import AutoModel
import timm
print("✓ transformers + timm import OK (PyTorch >= 2.4 confirmed)")

✓ Dependencies installed
✓ transformers + timm import OK (PyTorch >= 2.4 confirmed)


## 5 — Log in to Weights & Biases

**One-time setup — add your API key as a Colab secret:**
1. Click the 🔑 **key icon** in the left sidebar
2. Click **+ Add new secret**
3. Name: `WANDB_API_KEY` — Value: your key from [wandb.ai/authorize](https://wandb.ai/authorize)
4. Toggle **Notebook access** ON
5. Re-run this cell

In [6]:
import wandb
from google.colab import userdata

WANDB_KEY = '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'
assert WANDB_KEY, (
    "WANDB_API_KEY secret not found.\n"
    "Follow the steps in the cell above, then re-run."
)

wandb.login(key=WANDB_KEY, relogin=True)
print(f"✓ W&B logged in as: {wandb.api.default_entity}")
print(f"  Project URL: https://wandb.ai/{wandb.api.default_entity}/jit-rcdm")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: severinle (severinle-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ W&B logged in as: severinle-johannes-kepler-universit-t-linz
  Project URL: https://wandb.ai/severinle-johannes-kepler-universit-t-linz/jit-rcdm


## 6 — Copy RETFound checkpoint from Drive

In [7]:
import shutil, os

SRC = "/content/drive/MyDrive/jit_rcdm/retfound/RETFound_mae_natureCFP.pth"
DST = "/content/jit_rcdm/checkpoints/retfound/RETFound_mae_natureCFP.pth"

assert os.path.exists(SRC), (
    f"RETFound checkpoint not found at {SRC}.\n"
    "Upload RETFound_mae_natureCFP.pth to Google Drive at MyDrive/jit_rcdm/retfound/"
)

os.makedirs(os.path.dirname(DST), exist_ok=True)
if not os.path.exists(DST):
    print("Copying RETFound_mae_natureCFP.pth from Drive (~3.7 GB, this can take a few minutes)...")
    shutil.copy2(SRC, DST)
else:
    print(f"✓ RETFound checkpoint already at {DST}")

size_gb = os.path.getsize(DST) / 1e9
print(f"  size: {size_gb:.2f} GB")
print("✓ RETFound checkpoint present")

Copying RETFound_mae_natureCFP.pth from Drive (~3.7 GB, this can take a few minutes)...
  size: 3.95 GB
✓ RETFound checkpoint present


## 7 — Copy packed dataset from Drive

`train_packed_retfound.pt` stores both image tensors (uint8) **and** RETFound representations (1024-dim) in one file — no raw image folder needed on Colab.

**Create it locally first (run once on your machine):**
```bash
python scripts/precompute_reps_retfound.py \
    --data_dir   data/messidor2/train \
    --out_file   data/messidor2/train_reps_retfound.pt \
    --device     mps

python scripts/pack_dataset.py \
    --reps_file  data/messidor2/train_reps_retfound.pt \
    --out_file   data/messidor2/train_packed_retfound.pt \
    --image_size 224
```
Then upload `data/messidor2/train_packed_retfound.pt` (~150 MB) to Drive at `MyDrive/jit_rcdm/train_packed_retfound.pt`.

In [8]:
import shutil, os, torch

SRC = "/content/drive/MyDrive/jit_rcdm/train_packed_retfound.pt"
DST = "/content/jit_rcdm/data/messidor2/train_packed_retfound.pt"

assert os.path.exists(SRC), (
    f"train_packed_retfound.pt not found at {SRC}.\n"
    "Run precompute_reps_retfound.py + pack_dataset.py locally first, then upload to Google Drive."
)

os.makedirs(os.path.dirname(DST), exist_ok=True)
if not os.path.exists(DST):
    print("Copying train_packed_retfound.pt from Drive (~150 MB)...")
    shutil.copy2(SRC, DST)
else:
    print("train_packed_retfound.pt already present")

data = torch.load(DST, map_location="cpu", weights_only=False)
assert "images" in data and "reps" in data, "File missing 'images' or 'reps' key — re-run pack_dataset.py"
imgs, reps = data["images"], data["reps"]
assert reps.shape[1] == 1024, f"Expected 1024-dim RETFound reps, got {reps.shape[1]}"
assert imgs.dtype == torch.uint8, f"Expected uint8 images, got {imgs.dtype}"
print(f"✓ {len(imgs)} packed images {tuple(imgs.shape)}, dtype={imgs.dtype}")
print(f"  reps {tuple(reps.shape)}, mean norm={reps.norm(dim=1).mean():.2f}")
print("✓ Packed dataset OK")

Copying train_packed_retfound.pt from Drive (~150 MB)...
✓ 972 packed images (972, 3, 224, 224), dtype=torch.uint8
  reps (972, 1024), mean norm=34.59
✓ Packed dataset OK


## 8 — Verify full pipeline

> Re-run this cell after a `git pull` to reload updated code.

In [9]:
# Reload modules cleanly after a git pull
import importlib, sys
for mod in list(sys.modules.keys()):
    if mod.startswith("rcdm"):
        del sys.modules[mod]

import sys, os
sys.path.insert(0, "/content/jit_rcdm")

from rcdm.encoder_retfound import load_encoder, build_transform, RETFOUND_CHECKPOINT
from rcdm.jit          import JiT_S_16, FlowMatching, create_jit_model
from rcdm.dataset      import RepresentationDataset
from rcdm.conditioning import RMSNorm, AdaLNZero, ConditioningProjector
import torch

print("✓ All imports OK")
print(f"  RETFOUND_CHECKPOINT → {RETFOUND_CHECKPOINT}")
assert os.path.exists(RETFOUND_CHECKPOINT), f"Checkpoint not found at {RETFOUND_CHECKPOINT}"
print("  Checkpoint file exists ✓")

# Encoder smoke test (CPU) — load RETFound and run a forward pass
encoder = load_encoder(device="cpu")
n_enc = sum(p.numel() for p in encoder.parameters())
print(f"✓ RETFound ViT-L/16 loaded — {n_enc/1e6:.1f}M params")
with torch.no_grad():
    feats = encoder.forward_features(torch.randn(1, 3, 224, 224))
print(f"  CLS token shape: {tuple(feats[:, 0, :].shape)}")  # (1, 1024)

# Forward-pass smoke test (CPU) — JiT_S_16 with h_dim=1024
m = JiT_S_16(image_size=224, h_dim=1024)
m.eval()
with torch.no_grad():
    out = m(torch.randn(2,3,224,224), torch.rand(2), torch.randn(2,1024))
assert out.shape == (2,3,224,224)
n = sum(p.numel() for p in m.parameters())
print(f"✓ JiT_S_16 forward pass — output {tuple(out.shape)}, {n/1e6:.1f}M params")
print(f"  null_h  : shape={tuple(m.null_h.shape)}, requires_grad={m.null_h.requires_grad}")
print(f"  freqs_cis: shape={tuple(m.freqs_cis.shape)}, dtype={m.freqs_cis.dtype}")
print("\n✓ Pipeline ready — proceed to training")

✓ All imports OK
  RETFOUND_CHECKPOINT → /content/jit_rcdm/checkpoints/retfound/RETFound_mae_natureCFP.pth
  Checkpoint file exists ✓
✓ RETFound ViT-L/16 loaded — 303.3M params
  CLS token shape: (1, 1024)
✓ JiT_S_16 forward pass — output (2, 3, 224, 224), 25.8M params
  null_h  : shape=(1024,), requires_grad=True
  freqs_cis: shape=(196, 32), dtype=torch.complex64

✓ Pipeline ready — proceed to training


## 9 — Train

Checkpoints saved to `MyDrive/jit_rcdm/checkpoints_retfound/` every 5 000 steps — survive session disconnects.

In [11]:
import torch
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
batch  = 256 if mem_gb >= 70 else (128 if mem_gb >= 38 else 32)
print(f"GPU VRAM: {mem_gb:.0f} GB → batch_size={batch}")

GPU VRAM: 42 GB → batch_size=128


In [ ]:
import os
CKPT_DIR = "/content/drive/MyDrive/jit_rcdm/checkpoints_retfound"
os.makedirs(CKPT_DIR, exist_ok=True)

!python scripts/train.py \
    --model          S16 \
    --reps_file      data/messidor2/train_packed_retfound.pt \
    --save_dir       {CKPT_DIR} \
    --image_size     224 \
    --h_dim          1024 \
    --batch_size     {batch} \
    --grad_accum     1 \
    --lr             1e-4 \
    --warmup_steps   2000 \
    --total_steps    100000 \
    --save_interval  5000 \
    --log_interval   100 \
    --cfg_dropout    0.1 \
    --ema_decay      0.9999 \
    --wandb_project  jit-rcdm \
    --wandb_run_name S16-RETFound-100k-A100 \
    --device         cuda

## 9b — Resume after disconnection

Re-run cells 1–8, then run this cell. Latest checkpoint is picked up automatically.

In [12]:
import os, glob, torch
CKPT_DIR = "/content/drive/MyDrive/jit_rcdm/checkpoints_retfound"
mem_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
batch    = 256 if mem_gb >= 70 else (128 if mem_gb >= 38 else 32)

ckpts  = sorted(glob.glob(os.path.join(CKPT_DIR, "jit_rcdm_step*.pt")))
assert ckpts, f"No checkpoint found in {CKPT_DIR}"
latest = ckpts[-1]
print(f"Resuming from: {latest}")

!python scripts/train.py \
    --model          S16 \
    --reps_file      data/messidor2/train_packed_retfound.pt \
    --save_dir       {CKPT_DIR} \
    --image_size     224 \
    --h_dim          1024 \
    --batch_size     {batch} \
    --grad_accum     1 \
    --lr             1e-4 \
    --warmup_steps   2000 \
    --total_steps    100000 \
    --save_interval  5000 \
    --log_interval   100 \
    --cfg_dropout    0.1 \
    --ema_decay      0.9999 \
    --resume         {latest} \
    --wandb_project  jit-rcdm \
    --wandb_run_name S16-RETFound-100k-A100 \
    --device         cuda

Resuming from: /content/drive/MyDrive/jit_rcdm/checkpoints_retfound/jit_rcdm_step0095000.pt

[1/4] Building JiT model...
  Using preset JiT_S16
  Resuming from /content/drive/MyDrive/jit_rcdm/checkpoints_retfound/jit_rcdm_step0095000.pt
  Resuming from step 95000
  Total parameters  : 25.8M
  Trainable         : 25.8M

[2/4] Loading dataset...
Loading representations from data/messidor2/train_packed_retfound.pt...
  972 packed image-representation pairs loaded (no disk access at training time)
  972 samples, 7 batches/epoch at batch_size=128

[3/4] Setting up optimiser...
  EMA restored (decay=0.9999)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: severinle (severinle-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Wai

## 10 — Sample

| Steps trained | `--cfg_scale` |
|---|---|
| < 15 k | 1.0 |
| 15–30 k | 1.5 |
| 30–50 k | 2.0 |
| > 50 k | 2.0–3.0 |

In [13]:
import os, glob
CKPT_DIR   = "/content/drive/MyDrive/jit_rcdm/checkpoints_retfound"
SAMPLE_DIR = "/content/drive/MyDrive/jit_rcdm/samples_retfound"
os.makedirs(SAMPLE_DIR, exist_ok=True)

ckpts  = sorted(glob.glob(os.path.join(CKPT_DIR, "jit_rcdm_step*.pt")))
latest = ckpts[-1] if ckpts else os.path.join(CKPT_DIR, "jit_rcdm_final.pt")
print(f"Sampling from: {latest}")

TEST_IMAGES = sorted(glob.glob("/content/drive/MyDrive/jit_rcdm/test_images/*.png"))
assert TEST_IMAGES, "Upload .png test images to Drive at MyDrive/jit_rcdm/test_images/"
cond_args = " ".join(TEST_IMAGES)
print(f"Conditioning images: {len(TEST_IMAGES)}")

!python scripts/sampling_retfound.py \
    --checkpoint     {latest} \
    --cond_images    {cond_args} \
    --out_dir        {SAMPLE_DIR} \
    --n_samples      4 \
    --num_steps      50 \
    --cfg_scale      1.0 \
    --wandb_project  jit-rcdm \
    --wandb_run_name S16-RETFound-samples \
    --device         cuda

Sampling from: /content/drive/MyDrive/jit_rcdm/checkpoints_retfound/jit_rcdm_step0100000.pt
Conditioning images: 5
Loading JiT-RCDM from /content/drive/MyDrive/jit_rcdm/checkpoints_retfound/jit_rcdm_step0100000.pt...
  [EMA] loaded EMA weights for inference
  image_size=224, hidden_dim=384, cond_dim=128, h_dim=1024, trained_steps=100000
Loading RETFound encoder...
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: severinle (severinle-johannes-kepler-universit-t-linz) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run 9mhdqg5q (0.0s)
wandb: ⣻ setting up run 9mhdqg5q (0.0s)
wandb: ⣽ setting up run 9mhdqg5q (0.0s)
wandb: ⣾ setting up run 9mhdqg5q (0.0s)
wandb: ⣷ setting up run 9mhdqg5q (0.5s)
wandb: ⣯ setting up run 9mhdqg5q (0.5s)
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /conte

## 11 — Monitor

In [ ]:
!nvidia-smi
!ls -lh /content/drive/MyDrive/jit_rcdm/checkpoints_retfound/ 2>/dev/null || echo "No checkpoints yet"

---

## Drive layout

```
MyDrive/jit_rcdm/
    retfound/
        RETFound_mae_natureCFP.pth   ← upload before running (~3.7 GB)
    train_packed_retfound.pt         ← run precompute_reps_retfound.py + pack_dataset.py locally, then upload (~150 MB)
    test_images/                     ← upload before sampling (shared with DinoV3 notebook)
    checkpoints_retfound/            ← auto-created during training
    samples_retfound/                ← auto-created during sampling
```